# Diagnostics & verification — STURM ⇄ MESSAGEix-Austria linkage

Companion to `run.ipynb` (the results batch). Nothing here feeds `viz/` or the
thesis tables — these cells back **methodology claims and verification**: the
STURM smoke test, the bounds-mode (hard-linkage) probe and column
(coupling-mode comparison — since §17 the headline is `useful`, so **bounds** is
the sensitivity here), the exogenous-multiplier digitalization comparison
(mode-flip evidence, 2026-07-07; bounds-mode only), the once-through hand-off
(iterative-vs-once-through argument, methodology §3.5.1), and the
endogenous-CO₂ / W5.6 verification checks. Run cells on demand; each section
notes what it verifies. Same environment/licence requirements as `run.ipynb`.

In [ ]:
import logging, os, sys
from pathlib import Path

# --- ixmp platform. Set $SMX_PLATFORM to use a private database (recommended on a
# shared server: the shared default store corrupts under repeated open/close):
#   ixmp platform add <name> jdbc hsqldb /home/<you>/<name>-localdb/default
#   export SMX_PLATFORM=<name>
# Unset → the workbook's platform_name (blank = the ixmp local default database).
# NOTE: load_base_scenario imports the baseline FRESH for every cell (~25 s each;
# no state leaks between cells).
import ixmp
_plat = os.environ.get("SMX_PLATFORM")
if _plat:
    try:
        ixmp.config.get_platform_info(_plat)
    except ValueError:  # register it (ixmp config is per-environment, not part of the repo)
        _db = Path.home() / f"{_plat}-localdb" / "default"
        ixmp.config.add_platform(_plat, "jdbc", "hsqldb", str(_db))
        ixmp.config.save()
        print(f"registered ixmp platform {_plat!r} -> {_db}")

# make the package importable without `pip install -e .`
src = Path.cwd() / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))
logging.basicConfig(level=logging.INFO, force=True)

from common.config import Config
from linkage.pipeline import run_scenario, run_scenarios
from messageix import build
from sturm.driver import run_offline
from common.scenarios import Scenario, pathways, digitalization_levels
from linkage.runlog import start as start_runlog

# tee every INFO log (convergence trace, STURM year summary, warnings) to a file
LOGFILE = start_runlog()
print("run log ->", LOGFILE)

cfg = Config.load()   # reads data/inputs.xlsx (the single input source; no fallbacks)
# Useful-energy coupling with the r8 efficiency channel (the only path since 2026-09-13).
# the bounds-mode comparison cells below use dataclasses.replace. Fresh
# scenario_name (baseline-2025-r7, workbook) keeps the import hygiene.
assert cfg.scenario_name == "baseline-2025-r8", "stale inputs.xlsx — re-sync the repo"
assert cfg.carbon_price_passthrough, "W5.3 passthrough off — stale config"
assert cfg.price_damping < 1.0, "W5.8 price damping inactive — stale inputs.xlsx"
ref = pathways()["reference"]
print("ixmp platform:", cfg.platform_name)

# Open ONE ixmp platform and reuse it for every cell below. Opening a fresh
# platform per cell makes the HSQLDB file store fail to reopen partway through a
# batch (ClassCastException in the .script reload); one shared platform avoids it.
platform = build.get_platform(cfg)

# W5.8 probe gate helper (same as run.ipynb §2): the damped loop must CONVERGE,
# not fall back to the oscillation mean.
def run_verdicts(logfile):
    """Latest per-run-id convergence verdict from the runlog (True = converged)."""
    import re as _re
    _RUN = _re.compile(r"INFO (?:sturm_messageix|linkage)\.pipeline: Running (\S+)")
    verdicts, current = {}, None
    for line in Path(logfile).read_text(encoding="utf-8", errors="replace").splitlines():
        m = _RUN.search(line)
        if m:
            current = m.group(1)
        elif current and "Linkage converged" in line:
            verdicts[current] = True
        elif current and "Iteration budget" in line:
            verdicts[current] = False
    return verdicts

def probe_check(mode, rid):
    # Run-id-scoped (matches run.ipynb since 2026-08-28); the old global
    # last-line version was correct only by cell-ordering luck.
    verdicts = run_verdicts(LOGFILE)
    assert rid in verdicts, f"no convergence trace for {rid} — did the cell run?"
    assert verdicts[rid], f"{mode} probe did not converge — see the runlog"
    print(f"{rid}: converged")

print("config:", cfg)
print("ready")

## 1 — STURM smoke test (Austria)
Runs the vendored R model head-lessly for `C-AUT` (`cfg.sturm_region` — the only
region left since the W5.1 prune; the old `C-WEU-AUT` fallback is not part of this repository).
Check the report has all years (2025–2040) and the W5.6 carriers (`elec_hp`, `d_heat`).

In [ ]:
import pandas as pd
report = run_offline(region_select=[cfg.sturm_region], years=sorted({cfg.base_year, *cfg.years}))
df = pd.read_csv(report)
print("STURM report:", report)
print("final-energy years:", sorted(df.loc[df['level'] == 'final', 'year'].unique()))
fuels = sorted({c.rsplit('_', 1)[-1] if 'd_heat' not in c else 'd_heat' for c in df.loc[df['level'] == 'final', 'commodity']})
print("carriers:", fuels, "— expect elec_hp + d_heat present (W5.6)")
df.head()

## 3f — Diagnostics: once-through + free base year (validation)

**Once-through:** `max_iterations=1` (no price feedback — the Mastrucci-style
hand-off) → `…__once/` folder; diff vs the converged run quantifies the value of
iteration. (The free-base-year diagnostic script was retired with the refactor;
its `diagnostic-free_base_*` exports and the script live in pre-refactor git
history.)


In [ ]:
# --- 3f. Once-through diagnostic (reference/baseline, no price feedback) ---
import dataclasses

cfg_once = dataclasses.replace(cfg, max_iterations=1)
run_scenario(Scenario(pathway=ref, digitalization=digitalization_levels()["baseline"]),
             cfg_once, platform=platform)
print("once-through diagnostic done (…__once folder)")


### Endogenous vs post-hoc CO₂ (verification)

Cross-checks the in-model `EMISS` (`buildings_co2_endogenous.csv`, MtCO₂) against the post-hoc territorial calc (`buildings_co2.csv`, kt) — they should match within a few %. A large gap points at the `emission_factor` derivation or a missing bake (cell 1b).

In [ ]:
# --- Endogenous (in-model EMISS) vs post-hoc territorial CO2 (verification) ---
# Validates emission_factor = input_coef x EF x 8.76: the two should agree within a few %.
# buildings_co2_endogenous.csv is MtCO2; post-hoc buildings_co2.csv is kt -> x1000.
# NB: outputs written before 2026-07-04 show diff ~ +100% - that was the fixed
# World-node double count in the reporter, not a bake problem; fresh runs agree ~0%.
from common.paths import RESULTS_RUNS
import pandas as pd

for p in sorted(RESULTS_RUNS.glob("pathway-*__digi-*")):
    endo_f, post_f = p / "buildings_co2_endogenous.csv", p / "buildings_co2.csv"
    if not (endo_f.exists() and post_f.exists()):
        continue
    endo, post = pd.read_csv(endo_f), pd.read_csv(post_f)
    if endo.empty:
        print(f"{p.name}: endogenous empty - emission bake missing in baseline_4 (restore the baked workbook from the repository)")
        continue
    m = endo.rename(columns={"value": "endo_Mt"}).merge(
        post.rename(columns={"value": "post_kt"}), on="year")
    m["endo_kt"] = m["endo_Mt"] * 1000.0
    m["diff_%"] = (m["endo_kt"] - m["post_kt"]) / m["post_kt"] * 100
    print(f"\n{p.name}")
    print(m[["year", "endo_kt", "post_kt", "diff_%"]].round(2).to_string(index=False))

### W5.6 verification — endogenous district heat + explicit HP (coupled)

The standalone STURM gates passed on 2026-07-10; this checks the **coupled** side:
per-fuel base-year calibration with the restructured fuel set (the W5.6 revert rule
hangs on this), district heat differing across pathways — in useful mode this reflects the
dynamics-strip asymmetry (Reference keeps heat_rc's no-decline dynamics,
fossil-dial pathways strip them), not an endogenous price response — and a
live resistive/heat-pump split.

In [ ]:
# --- W5.6 coupled verification: base year + DH differentiation + HP split ---
import pandas as pd
from common.paths import RESULTS_RUNS

byc = pd.read_csv(RESULTS_RUNS / "pathway-reference__digi-baseline" / "base_year_check.csv")
print(byc.to_string(index=False))
failed = byc[~byc.passed]  # coal exemption removed 2026-08-28: the useful-mode
# anchor pins coal_rc from the reference, so coal passes and the old
# exemption only masked future failures
assert failed.empty, f"BASE-YEAR GATE FAILED (W5.6 revert rule): {list(failed.fuel)}"

def _act(pathway, tech, year=2040):
    a = pd.read_csv(RESULTS_RUNS / f"pathway-{pathway}__digi-baseline" / "activity_rc.csv")
    v = a[(a.year == year) & (a.technology == tech)]["value"]
    return float(v.iloc[0]) if not v.empty else float("nan")

dh = {p: _act(p, "heat_rc") for p in ("reference", "renewables_push", "bio_bridge")}
print("\nheat_rc (district heat) 2040 by pathway:", {k: round(v, 4) for k, v in dh.items()})
# NOTE (2026-08-28): this difference is caused by the dynamics strip (Reference
# keeps heat_rc's no-decline dynamics -> pinned; fossil-dial pathways strip
# them -> free), NOT by an endogenous DH price response. It verifies the
# strip's asymmetry, no more.
assert len({round(v, 3) for v in dh.values()}) > 1, "DH identical across pathways — strip asymmetry gone?"

hp, res = _act("reference", "hp_el_rc"), _act("reference", "elec_rc")
print(f"reference 2040: hp_el_rc {hp:.4f} vs elec_rc {res:.4f} (explicit split live)")
print("W5.6 coupled verification passed")

## §4x. Degeneracy probe (r7 B10)

The r6 RP cells showed `gas_rc` identical to 4 dp across different demand levels
while *below* its cap at 2030 — the signature of alternate optima. Re-solve the
RP baseline cell with dual simplex (`lpmethod=2` instead of barrier 4) under a
probe id and compare: if `gas_rc` moves, the "zero RQ2 spread" at non-binding
years is a degenerate-vertex artifact and must be reported as a range.

In [ ]:
# --- degeneracy probe: same cell, different LP algorithm ---
import dataclasses
import pandas as pd
from common.paths import RESULTS_RUNS

rp = pathways()["renewables_push"]
probe = dataclasses.replace(rp, id="renewables_push_lpprobe",
                            label="RP baseline (dual simplex probe)")
cfg_probe = dataclasses.replace(cfg, solve={"model": "MESSAGE",
                                            "solve_options": {"lpmethod": "2"}})
run_scenario(Scenario(pathway=probe, digitalization=digitalization_levels()["baseline"]),
             cfg_probe, platform=platform)

a0 = pd.read_csv(RESULTS_RUNS / "pathway-renewables_push__digi-baseline" / "activity_rc.csv")
a1 = pd.read_csv(RESULTS_RUNS / "pathway-renewables_push_lpprobe__digi-baseline" / "activity_rc.csv")
for y in (2030, 2035, 2040):
    g0 = a0[(a0.technology == "gas_rc") & (a0.year == y)]["value"].iloc[0]
    g1 = a1[(a1.technology == "gas_rc") & (a1.year == y)]["value"].iloc[0]
    flag = "  <-- alternate optimum" if abs(g0 - g1) > 1e-4 else ""
    print(f"gas_rc {y}: barrier {g0:.4f} vs dual-simplex {g1:.4f}{flag}")